# What a leak would have looked like

A forecast for day D is made at 09:00 on D−1 and may read only what was known then. The pipeline holds every feature to that with an as-of filter, a rule that checks every feature query for it, and tests that plant values from the future. This notebook asks what those guards are worth: **if a feature had read the future, how much better would the backtest have looked, and what would have caught it?**

It is the experiment behind the first fault scenario, [F2](../docs/scenarios/F2-lookahead-leak.md). The expected outcome was written there before anything here was run, and is the "We assumed" column of the table at the end.

Two leaks are injected:

| Leak | The features read |
| --- | --- |
| Lags | The same slot's output over D−7 to D−1 at its latest version, where the pipeline reads D−8 to D−2 as known at the decision time |
| Weather | The forecast one day old, and the same day's, where the pipeline reads the two-day-old one |

**Nothing here touches the pipeline.** The leaky features are built by this notebook's own queries and never written anywhere. The model is trained and scored on them with the pipeline's code (`jtf.ml`), on the 16 months the honest result stands on, March 2025 to June 2026.

## 0. The honest model

**Step.** The rows an honest run is trained on and predicts from, read as a run reads them, and the model trained month by month on them as a run trains it. The scoring is notebook 11's, which agrees with recorded runs.

In [1]:
import os
import sys
import time
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from jtf.ml.backtest import QUANTILES, by_group, metrics
from jtf.ml.models import ALPHAS, KEPT_INPUTS, columns_of, train_boosted
from jtf.ml.replay import train_by_month
from jtf.ml.warehouse import BacktestWarehouse

warnings.filterwarnings("ignore", message="BigQuery Storage module not found")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 70)
pd.set_option("display.float_format", "{:,.4f}".format)


def setting(name: str) -> str:
    if value := os.environ.get(name):
        return value
    lines = Path("../.env").read_text().splitlines()
    env = dict(line.split("=", 1) for line in lines if "=" in line and not line.startswith("#"))
    return env[name]


PROJECT, PREFIX = setting("JTF_BIGQUERY_PROJECT"), setting("JTF_DBT_DATASET_PREFIX")
warehouse = BacktestWarehouse(PROJECT, setting("JTF_BIGQUERY_LOCATION"), PREFIX)
DWD, DWS, ADS = (f"{PROJECT}.{PREFIX}_{layer}" for layer in ("dwd", "dws", "ads"))
JST = "Asia/Tokyo"


def sql(query: str) -> pd.DataFrame:
    return warehouse.client.query(query).to_dataframe(create_bqstorage_client=False)


FIRST_TEST, LAST = date(2025, 3, 1), date(2026, 6, 30)  # the months the honest 5.21% stands on

# What an honest run is trained on and predicts from, read as a run reads it
columns = tuple(dict.fromkeys([*columns_of(KEPT_INPUTS), "solar_d2_mw"]))
known = warehouse.training(LAST, columns)
days = sql(f"""
    SELECT day AS target_day, sky, is_curtailed AS is_curtailed_day, season
    FROM `{ADS}.ads_ml_day_solar`
""")
known = known.merge(days, on="target_day", how="left", validate="m:1")
known["fold"] = pd.to_datetime(known["target_day"]).dt.to_period("M").dt.start_time.dt.date
in_test = known["fold"] >= FIRST_TEST


def outcomes(said: pd.DataFrame) -> pd.DataFrame:
    """Forecasts for the test rows met by their labels, as ads_ml_outcome_solar gives a run's
    (notebook 11, section 2, sets this beside recorded runs: they agree to 5e-12)."""
    test = known[in_test]
    said = said.reindex(columns=QUANTILES)
    label, capacity = test["label_mw"], test["capacity_mw"]
    error = said["predicted_mw"] - label
    miss = -said.sub(label, axis=0)
    levels = pd.Series(ALPHAS)
    pinball = np.maximum(miss * levels, miss * (levels - 1)).sum(axis=1, skipna=False) / 3
    kept = [
        "fold",
        "decision_time",
        "target_day",
        "slot_start",
        "is_daylight",
        "sky",
        "is_curtailed_day",
        "season",
    ]
    return test[kept].assign(
        error_mw=error,
        error_share=error / capacity,
        in_range=label.between(said["p10_mw"], said["p90_mw"]).where(said["p10_mw"].notna()),
        pinball_share=pinball / capacity,
    )


def read(said: pd.DataFrame) -> dict[str, float]:
    """A forecast's miss overall and by sky, its coverage, and its miss per test month."""
    scored = outcomes(said)
    per_fold, overall = metrics(scored)
    sky = by_group(scored).query("grouping == 'sky'").set_index("group")["mae_share"]
    return {
        "mae_share": overall["mae_share"],
        "clear": sky["clear"],
        "mixed": sky["mixed"],
        "cloudy": sky["cloudy"],
        "coverage": overall.get("coverage", np.nan),
        "by_month": per_fold.set_index("fold")["mae_share"],
    }


def model(rows: pd.DataFrame) -> pd.DataFrame:
    """The model trained month by month on `rows` and its forecasts for their test months."""
    return train_by_month(train_boosted, rows, rows[in_test])[0]


started = time.time()
RESULTS = {"honest": read(model(known))}
print(
    f"{len(known):,} labelled slots to {known['target_day'].max()}; "
    f"{int(in_test.sum()):,} in {known.loc[in_test, 'fold'].nunique()} test months"
)
print(
    f"the honest model, trained and scored here in {time.time() - started:.0f} s: "
    f"{RESULTS['honest']['mae_share']:.4f} of capacity"
)

41,520 labelled slots to 2026-06-30; 23,376 in 16 test months
the honest model, trained and scored here in 7 s: 0.0521 of capacity


**What it shows.** **Observed:** 5.21% of capacity, the figure of the recorded backtest. Everything below is measured against it, with the same code and the same months.

## 1. The lag leak

**Step.** One query builds the leaky lags: the same slot on each of the seven days D−7 to D−1, at its latest version, with no as-of filter. Beside each value it counts whether it was known at the slot's decision time. The seven-day mean in the honest rows is then replaced by the leaky one and the model trained again. The two baselines that are nothing but a lag are scored both ways too: persistence (one day back instead of two) and the seven-day mean.

In [2]:
# The lag leak: the same slot over D-7 to D-1 at its latest version, with no as-of filter, and
# beside each value read, whether it was known at the slot's decision time
LAG_LEAK = f"""
    SELECT s.decision_time, s.slot_start,
      IF(COUNT(a.value_mw) = 14, SUM(a.value_mw) / 7, NULL) AS leaky_mean_7d_mw,
      SUM(IF(days_back = 1, a.value_mw, NULL)) AS leaky_d1_mw,
      COUNTIF(a.known_at > s.decision_time) AS read_but_not_known_at_t,
      COUNTIF(days_back = 1 AND a.known_at > s.decision_time) AS of_them_from_d1,
      COUNT(a.value_mw) AS read
    FROM `{DWS}.dws_decision_slots` AS s
    CROSS JOIN UNNEST(GENERATE_ARRAY(1, 7)) AS days_back
    INNER JOIN `{DWD}.dwd_tepco_actuals` AS a
      ON a.slot_start = TIMESTAMP_SUB(s.slot_start, INTERVAL days_back DAY)
      AND a.measure IN ('solar', 'solar_curtailed')
      AND a.known_until IS NULL
    GROUP BY s.decision_time, s.slot_start
"""
lags = sql(LAG_LEAK)
leaky = known.merge(lags, on=["decision_time", "slot_start"], how="left", validate="1:1")
leaky.index = known.index
test = leaky[in_test & leaky["is_daylight"].astype(bool)]
unknown, from_d1 = test["read_but_not_known_at_t"].sum(), test["of_them_from_d1"].sum()
print(
    f"values read for the test months' daylight slots: {int(test['read'].sum()):,}; "
    f"not yet known at the decision time: {int(unknown):,} "
    f"({unknown / test['read'].sum():.1%}), {int(from_d1):,} of them from D-1"
)
changed = (leaky["leaky_mean_7d_mw"] - leaky["solar_mean_7d_mw"]).abs() > 1e-6
print(f"slots whose seven-day mean differs from the honest one: {changed[in_test].mean():.1%}")

RESULTS["lags: the model"] = read(model(leaky.assign(solar_mean_7d_mw=leaky["leaky_mean_7d_mw"])))
baselines = {
    "persistence, honest (D-2)": known.loc[in_test, "solar_d2_mw"],
    "persistence, leaky (D-1)": leaky.loc[in_test, "leaky_d1_mw"],
    "seven-day mean, honest": known.loc[in_test, "solar_mean_7d_mw"],
    "seven-day mean, leaky": leaky.loc[in_test, "leaky_mean_7d_mw"],
}
RESULTS |= {name: read(said.to_frame("predicted_mw")) for name, said in baselines.items()}
pd.DataFrame(RESULTS).T.drop(columns="by_month")

values read for the test months' daylight slots: 177,128; not yet known at the decision time: 19,836 (11.2%), 19,836 of them from D-1
slots whose seven-day mean differs from the honest one: 52.7%


,mae_share,clear,mixed,cloudy,coverage
honest,0.0521,0.0322,0.0585,0.0726,0.7311
lags: the model,0.0518,0.0320,0.0584,0.0719,0.7277
"persistence, honest (D-2)",0.1264,0.0960,0.1219,0.1778,NaN
"persistence, leaky (D-1)",0.1141,0.0893,0.1104,0.1559,NaN
"seven-day mean, honest",0.1052,0.0965,0.0692,0.1683,NaN
"seven-day mean, leaky",0.1027,0.0939,0.0690,0.1629,NaN


**What it shows.** **Observed:**

- **The leak is real:** of the 177,128 values read for the scored slots, 19,836 (11.2%) were not yet known at the decision time. All of them are from D−1, and they are 78% of D−1's values: at 09:00 that day has barely begun
- **It moves the feature:** the seven-day mean differs from the honest one in 52.7% of slots
- **The model's score hardly changes:** 5.18% against 5.21%, better in 7 months of 16, which is a coin's toss
- **The baselines gain what was predicted:** persistence 11.41% against 12.64%, the seven-day mean 10.27% against 10.52%

**Learned → design:**

- **A leak need not show in the score.** One value in nine came from the future and the model's backtest looks the same, because the model barely uses the past week ([notebook 11](11_inputs_of_the_solar_model.ipynb), section 4.2). Comparing a leaky backtest with an honest one would not have found this leak in this model. Only a check of the query itself does
- **No later correction was read:** every future value is from D−1. The history holds one version of each past actual, dated by its rule, so the case of a value corrected weeks later can't be measured on it

## 2. What the as-of filter does to a fresher forecast

**Step.** The forecast history holds each hour at three leads: issued the same day (lead 0), one day before (lead 1) and two days before (lead 2). One query builds the area's forecast at each lead as `dws_weather_area` builds it, the nine points weighted by capacity, at the latest version and with no filter. Lead 2 built this way is first set beside the forecast the pipeline reads, to see that the construction is the pipeline's. Then, per lead, how many daylight slots were known at the decision time at all nine points.

In [3]:
# MSM's forecast for the area at each lead the history holds, the nine points weighted as
# dws_weather_area weights them, at the latest version and with no as-of filter; and beside it,
# at how many of the nine points the value was known at the decision time
VARIABLES = ["ghi_w_m2", "direct_w_m2", "diffuse_w_m2", "cloud_cover_pct", "temperature_c"]
area = ", ".join(f"IF(COUNT(w.{v}) = 9, SUM(p.weight * w.{v}), NULL) AS {v}" for v in VARIABLES)
WEATHER_LEAK = f"""
    SELECT s.decision_time, s.slot_start, w.lead_days, {area},
      COUNTIF(w.known_at <= s.decision_time) AS points_known_at_t
    FROM `{DWS}.dws_decision_slots` AS s
    INNER JOIN `{DWD}.dwd_weather_slots` AS w
      ON w.slot_start = s.slot_start AND w.source = 'previous_runs' AND w.model = 'jma_msm'
      AND w.known_until IS NULL
    INNER JOIN `{DWS}.dws_point_weights` AS p
      ON p.decision_time = s.decision_time AND p.point = w.point
    WHERE s.target_day BETWEEN '2024-02-17' AND '{LAST}'
    GROUP BY s.decision_time, s.slot_start, w.lead_days
"""
leads = sql(WEATHER_LEAK)
by_lead = {
    lead: rows.set_index(["decision_time", "slot_start"])
    for lead, rows in leads.groupby("lead_days")
}
keys = pd.MultiIndex.from_frame(known[["decision_time", "slot_start"]])
lit = known["is_daylight"].astype(bool).to_numpy()

# Lead 2 built this way should be the forecast the pipeline already reads
two = by_lead[2].reindex(keys)
gap = (two["ghi_w_m2"].to_numpy() - known["forecast_ghi_w_m2"].to_numpy())[lit]
print(
    f"lead 2 rebuilt here against the pipeline's forecast, daylight slots: "
    f"{np.isnan(gap).mean():.2%} missing here, largest difference {np.nanmax(np.abs(gap)):.4f} W/m2"
)

known_at_t = pd.DataFrame(
    {
        f"lead {lead}": by_lead[lead].reindex(keys)["points_known_at_t"].to_numpy()[lit] == 9
        for lead in (0, 1, 2)
    }
)
hour = known.loc[lit, "slot_start"].dt.tz_convert(JST).dt.hour.to_numpy()
print("share of daylight slots whose forecast was known at the decision time, at all nine points:")
print((known_at_t.mean() * 100).round(1).to_dict())
(known_at_t.groupby(hour).mean() * 100).round(0).T

lead 2 rebuilt here against the pipeline's forecast, daylight slots: 0.04% missing here, largest difference 0.0000 W/m2
share of daylight slots whose forecast was known at the decision time, at all nine points:
{'lead 0': 0.0, 'lead 1': 6.4, 'lead 2': 100.0}


,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
lead 0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
lead 1,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
lead 2,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000


**What it shows.** **Observed:**

- **Lead 2 rebuilt here is the pipeline's forecast,** with no difference in any slot that has one. 0.04% of daylight slots have none here: there the pipeline fills from another weather model
- **At the decision time every daylight slot's two-day-old forecast was known**
- **Of the one-day-old forecast, 6.4% was known:** the slots before 06:00, and nothing after
- **Of the same day's, none**

**Learned → design:** read through the as-of filter, the fresher forecast is no leak. It is simply not there: from 06:00 on, every daylight slot of the day would have no weather, and the pipeline's test that every daylight slot has weather at the decision time would fail on the first build. To leak, a query has to go around the filter, or the forecast has to carry a date earlier than the truth.

## 3. The weather leak

**Step.** The day's forecast in the honest rows (total, direct and diffuse radiation, cloud cover, temperature) is replaced by the one-day-old forecast, then by the same day's, wherever the history has one, and the model trained again each time. A last run takes both leaks at once.

In [4]:
def with_weather(lead: int) -> pd.DataFrame:
    """The honest rows with the day's forecast taken from `lead`, wherever the history has it."""
    fresher = by_lead[lead].reindex(keys)
    swapped = {
        f"forecast_{v}": pd.Series(fresher[v].to_numpy(), index=known.index).fillna(
            known[f"forecast_{v}"]
        )
        for v in VARIABLES
    }
    return known.assign(**swapped)


for lead, name in ((1, "weather: one day old"), (0, "weather: the same day's")):
    rows = with_weather(lead)
    moved = (rows["forecast_ghi_w_m2"] - known["forecast_ghi_w_m2"]).abs() > 1e-6
    print(
        f"{name}: forecast radiation differs from the honest one in "
        f"{moved[in_test & lit].mean():.1%} of the test months' daylight slots"
    )
    RESULTS[name] = read(model(rows))
both = with_weather(0).assign(solar_mean_7d_mw=leaky["leaky_mean_7d_mw"])
RESULTS["both: lags and the same day's weather"] = read(model(both))

table = pd.DataFrame(RESULTS).T
honest = RESULTS["honest"]
models = table.loc[[name for name in table.index if ":" in name or name == "honest"]].copy()
models["better by"] = honest["mae_share"] - models["mae_share"]
models["months better"] = [
    int((result["by_month"] < honest["by_month"]).sum()) for result in models.to_dict("records")
]
models.drop(columns="by_month")

weather: one day old: forecast radiation differs from the honest one in 98.0% of the test months' daylight slots


weather: the same day's: forecast radiation differs from the honest one in 98.3% of the test months' daylight slots


,mae_share,clear,mixed,cloudy,coverage,better by,months better
honest,0.0521,0.0322,0.0585,0.0726,0.7311,0.0000,0
lags: the model,0.0518,0.0320,0.0584,0.0719,0.7277,0.0003,7
weather: one day old,0.0434,0.0287,0.0511,0.0543,0.7204,0.0087,15
weather: the same day's,0.0295,0.0201,0.0393,0.0296,0.7211,0.0226,16
both: lags and the same day's weather,0.0295,0.0202,0.0393,0.0297,0.7247,0.0226,16


**What it shows.** **Observed**, against the honest 5.21%:

| Leak | Miss | Better by | Months better | Cloudy days |
| --- | --- | --- | --- | --- |
| Lags | 5.18% | 0.03 | 7 of 16 | 7.19% |
| Weather one day old | 4.34% | 0.87 | 15 of 16 | 5.43% |
| The same day's weather | 2.95% | 2.26 | 16 of 16 | 2.96% |
| Both | 2.95% | 2.26 | 16 of 16 | 2.97% |

- **The same day's weather nearly halves the error.** On cloudy days it falls from 7.26% to 2.96%: with that forecast a cloudy day is no harder than a clear one was
- **The range looks no different:** it holds 72 to 73% in every run

**Learned → design:**

- **This is the leak that matters.** A backtest that read it would have shown a model with 2.95%, and the same model would miss by 5.2% once live
- **It also sizes what is left to gain.** Nearly all of the model's remaining error on cloudy days is the two-day-old forecast's own error: a forecast one day fresher is worth 0.9 points, and the plan's live capture, about a day old at the decision time, is the honest way to some of it

## 4. What the rule says of these queries

**Step.** CI runs a rule over every feature query the pipeline compiles: each read of a table that keeps versions must be filtered by the decision time, written as the as-of macro writes it (`warehouse/dbt/rules/test_as_of.py`). The same function is called here on the two leaky queries of this notebook, and on the lag query with the filter put in. That filtered query is then run, to see what the leak's window gives when read honestly.

In [5]:
# The rule CI runs over every feature query, on the two queries above: which decision time each
# read of a versioned table is filtered by. And the lag query again with the filter as the
# pipeline's macro writes it
sys.path.insert(0, "../warehouse/dbt/rules")
from test_as_of import reads  # noqa: E402

AS_OF = (
    "(a.known_at <= s.decision_time AND (a.known_until IS NULL OR s.decision_time < a.known_until))"
)
FILTERED = LAG_LEAK.replace("AND a.known_until IS NULL", f"AND {AS_OF}")
checked = {
    "the lag leak": LAG_LEAK,
    "the weather leak": WEATHER_LEAK,
    "the lag query with the as-of filter": FILTERED,
}
verdicts = pd.DataFrame(
    [
        {"query": name, "reads": found.table, "as of": found.t or "nothing: the rule fails"}
        for name, query in checked.items()
        for found in reads(query, {"dwd_tepco_actuals", "dwd_weather_slots"})
    ]
)
filtered = known.merge(
    sql(FILTERED), on=["decision_time", "slot_start"], how="left", validate="1:1"
)
filtered.index = known.index
scored = filtered[in_test & lit]
whole, unknown = scored["leaky_mean_7d_mw"].notna().mean(), scored["read_but_not_known_at_t"].sum()
print(
    f"with the filter, D-7 to D-1 gives a seven-day mean for {whole:.1%} of the test months' "
    f"daylight slots, and reads {int(unknown)} values not known at the decision time"
)
verdicts

with the filter, D-7 to D-1 gives a seven-day mean for 21.6% of the test months' daylight slots, and reads 0 values not known at the decision time


,query,reads,as of
0,the lag leak,dwd_tepco_actuals,nothing: the rule fails
1,the weather leak,dwd_weather_slots,nothing: the rule fails
2,the lag query with the as-of filter,dwd_tepco_actuals,s.decision_time


**What it shows.** **Observed:**

- **Both leaky queries fail the rule:** each reads a versioned table filtered by no decision time
- **With the filter the query passes,** and reads no value that wasn't known
- **And then it has little to give:** a seven-day mean over D−7 to D−1 exists for 21.6% of the scored slots, those of the early morning. That is why the pipeline's window is D−8 to D−2

**Learned → design:** the rule sees a leak where the score can't (section 1), and sees it before any data is read: in CI it runs on every pull request that touches the warehouse, and a failure blocks the merge. What it can't see is a value that passes the filter with a date earlier than the truth. That is held elsewhere: by the tests of each source's dating rule, and by the end-to-end test that plants absurd values known just after a decision time and requires that nothing at that time changes (`e2e_tests/test_time_machine.py`).

## What this exploration made known

| We assumed | We found | So |
| --- | --- | --- |
| The lag leak improves the model by less than 0.2 points | 0.03 points, better in 7 months of 16, though 11.2% of what it read was from the future (section 1) | Right, and more than right: **the score can't see this leak.** The plan lists the gap between a leaky and an honest backtest among the things that catch a leak; for an input the model barely uses, it doesn't |
| It improves persistence by 1 to 2 points and the seven-day mean by a few tenths | 1.23 points and 0.25 (section 1) | Right. A leak shows in a model in proportion to how much the model leans on the leaked input |
| The one-day-old forecast improves the model by 0.3 to 0.8 points | 0.87 points, in 15 months of 16 (section 3) | Slightly more than expected |
| The same day's improves it by 1 to 2 points, most on cloudy days | 2.26 points: 2.95% against 5.21%; cloudy days 2.96% against 7.26% (section 3) | **More than expected.** A leaked forecast would have nearly halved the reported error |
| The lag leak is caught by the rule over feature queries | It fails the rule: a versioned table read with no decision time (section 4) | Right. Caught before any data is read |
| The weather leak, read through the filter, is no leak: the fresher forecast isn't there | 6.4% of the one-day-old forecast's daylight slots were known at the decision time, none of the same day's (section 2) | Right. The filter turns the leak into missing weather, which a test reports |
| The weather leak, read around the filter, is caught by the same rule | It fails it the same way (section 4) | Right |
| A value dated too early is the case no rule over SQL can see | Not tested here: the history's dates are the rules themselves | It stays the residual risk. Every run says how many of its decision times rest on a rule: all 487 of the backtest's |

## Still open

| What | Why it can't be known yet | When |
| --- | --- | --- |
| What a late correction of an actual would leak | The history holds one version per past value | Once captured corrections exist: the first monthly re-capture, on 1 December |
| Whether the dating rules themselves are right | A rule can only be checked against captures | With every month of live capture, where values are dated by their fetch |

## Where the sources and the data disagree

| Where | Says | We found | Handled by |
| --- | --- | --- | --- |
| [Phase-1 plan](../docs/phase1/README.md#fault-scenarios), F2, "should be caught by" | "the gap between leaky and as-of backtests" | For the lag leak there is no gap to see: 0.03 points (section 1) | The plan names the rule and the planted-values test as what catches a leak; the gap is what a leak costs, where it shows |
| The same, F2's injection | Run-config presets on the replay job | The leaks were built in this notebook's queries instead, so that no switch for a leak exists in the pipeline's SQL | Agreed on 2026-11-20; the plan says so |

## Sources

| Page | Supports |
| --- | --- |
| [F2's write-up](../docs/scenarios/F2-lookahead-leak.md) | The expected outcome, written before this was run |
| [Notebook 11](11_inputs_of_the_solar_model.ipynb), sections 2 and 4.2 | The scoring used here; how little the model uses the past week |
| [Notebook 04](04_ods_to_dwd.ipynb), section 3.1 | Why the two-day-old forecast is the one known at the decision time |
| `warehouse/dbt/rules/test_as_of.py` | The rule run in section 4 |
| `e2e_tests/test_time_machine.py` | The test that plants values known after a decision time |